In [1]:
# Using Lucas-Kanade function

In [2]:
import cv2
import numpy as np

In [10]:
corner_track_params = dict(maxCorners=10, qualityLevel=0.03, minDistance=7, blockSize=7)

In [11]:
lk_params = dict(winSize=(200,200), maxLevel=2, criteria = (cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT,10,0.03))

In [16]:
cap = cv2.VideoCapture(0)

ret, prev_frame = cap.read()
prev_frame_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

# pts to track
prevPts = cv2.goodFeaturesToTrack(prev_frame_gray, mask = None, **corner_track_params)

mask = np.zeros_like(prev_frame)

while True:
    ret, curr_frame = cap.read()
    curr_frame_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
    
    # if flow found in corresponding feature, status will be 1, else 0
    nextPts, status, err = cv2.calcOpticalFlowPyrLK(prev_frame_gray, curr_frame_gray, prevPts,None, **lk_params) # nextPts = Nune here
    
    good_prev = prevPts[status == 1] 
    good_new = nextPts[status == 1]
    
    for i, (prev, new) in enumerate(zip(good_prev, good_new)):
        
        x_prev, y_prev = prev.ravel()
        x_new, y_new = new.ravel()
        
        mask = cv2.line(mask, (x_prev, y_prev), (x_new, y_new), (0,255,0), 2)
        
        curr_frame = cv2.circle(curr_frame, ( x_new, y_new), 8, (0,0,255), -1)
        
    img = cv2.add(curr_frame, mask)
    cv2.imshow('Tracking', img)
    
    k = cv2.waitKey(30) & 0xFF
    if k == 27:
        break
        
    prev_frame_gray = curr_frame_gray.copy()
    prevPts = good_new.reshape(-1,1,2)
        
cap.release()
cv2.destroyAllWindows()